# Tahap 4 : Case Solution Reuse
## Alur Kerja
1. **Muat artefak Tahap 3** — model, vectorizer, embeddings, train_df
2. **Ekstrak Solusi** — bangun `case_solutions`: `{case_id: solusi_text}` dari kolom `argumen_hukum` / `vonis_penjara`
3. **Algoritma Prediksi** — dua strategi:
   - *Majority vote*: pilih solusi (pasal) yang paling banyak muncul di top-k
   - *Weighted similarity*: bobot = skor cosine-similarity
4. **Fungsi `predict_outcome()`** — wrapper yang memanggil `retrieve()` lalu `voting/weighting`
5. **Demo Manual** — 5 contoh kasus baru → bandingkan prediksi vs putusan sebenarnya
6. **Simpan output** — `data/results/predictions.csv`


## 0. Setup & Path

In [ ]:
!pip install scikit-learn pandas numpy joblib transformers torch sentencepiece tqdm


In [15]:
import re
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR     = Path(".").resolve().parent   # project root (1 level di atas /notebooks)
MODELS_DIR   = BASE_DIR / "data" / "models"
PROCESSED_DIR= BASE_DIR / "data" / "processed"
EVAL_DIR     = BASE_DIR / "data" / "eval"
RESULTS_DIR  = BASE_DIR / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Models dir  : {MODELS_DIR}")
print(f"Results dir : {RESULTS_DIR}")


Models dir  : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\models
Results dir : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\results


## 1. Muat Artefak Tahap 3

Semua model dan data yang dibutuhkan dimuat dari disk (hasil simpan di akhir Tahap 3),
sehingga Tahap 4 tidak perlu re-train apapun.


In [16]:
# ── Jalur A: TF-IDF + ML models ──────────────────────────────────────────────
tfidf_vectorizer = joblib.load(MODELS_DIR / "tfidf_vectorizer.joblib")
svm_model        = joblib.load(MODELS_DIR / "svm_model.joblib")
nb_model         = joblib.load(MODELS_DIR / "nb_model.joblib")

# ── Jalur B: BERT embeddings ──────────────────────────────────────────────────
train_embeddings = np.load(MODELS_DIR / "train_embeddings.npy")

# ── Train set (case base index) ───────────────────────────────────────────────
train_df = pd.read_csv(MODELS_DIR / "train_df.csv")

# ── Full cases (untuk ambil solusi) ───────────────────────────────────────────
cases_df = pd.read_csv(PROCESSED_DIR / "cases.csv")

print(f"Train set    : {len(train_df)} kasus")
print(f"Full cases   : {len(cases_df)} kasus")
print(f"Embedding shape: {train_embeddings.shape}")


Train set    : 24 kasus
Full cases   : 30 kasus
Embedding shape: (24, 768)


In [17]:
def remove_admin_identity_lines(text: str) -> str:
    """
    Hapus baris identitas administratif Terdakwa (Nama, Tempat Lahir, dst.)
    agar representasi vektor fokus ke fakta perbuatan.
    (Fungsi identik dengan Tahap 3 — disalin agar notebook mandiri.)
    """
    field_label_pattern = re.compile(
        r"^\s*(?:I+\.?\s*|U\.?\s*)?"
        r"(?:Nama\s+[Ll]engkap|Tempat\s+[Ll]ahir|Umur\s*/\s*[Tt]anggal\s+[Ll]ahir|"
        r"Jenis\s+[Kk]elamin|Kebangsaan|Tempat\s+[Tt]inggal|Agama|Pekerjaan|Pendidikan)"
        r"\s*:?\s*[\w\s\-/\.,]{0,3}\s*$",
        flags=re.IGNORECASE | re.MULTILINE,
    )
    cleaned = field_label_pattern.sub("", text)
    return re.sub(r"\n{3,}", "\n\n", cleaned)


## 2. Rebuild Fungsi Retrieval

Fungsi `retrieve_tfidf()` dan `retrieve_embedding()` dibangun ulang menggunakan
artefak yang sudah dimuat, **mengembalikan juga skor similarity** agar bisa
dipakai oleh algoritma *weighted similarity* di langkah berikutnya.


In [18]:
def retrieve_tfidf(query: str, k: int = 5) -> list[tuple[str, float]]:
    """
    Retrieval TF-IDF + cosine similarity.
    Returns: list of (case_id, similarity_score), diurutkan dari tertinggi.
    """
    query_clean  = remove_admin_identity_lines(query)
    query_vector = tfidf_vectorizer.transform([query_clean])
    sims         = cosine_similarity(query_vector,
                       tfidf_vectorizer.transform(train_df["text_for_vector"]
                           if "text_for_vector" in train_df.columns
                           else train_df["ringkasan_fakta"].fillna("") + " " +
                                train_df["argumen_hukum"].fillna("")
                       )).flatten()
    top_idx = np.argsort(sims)[::-1][:k]
    return [(train_df.iloc[i]["case_id"], float(sims[i])) for i in top_idx]


def retrieve_embedding(query: str, k: int = 5) -> list[tuple[str, float]]:
    """
    Retrieval IndoBERT embedding + cosine similarity.
    Returns: list of (case_id, similarity_score), diurutkan dari tertinggi.
    """
    import torch
    from transformers import AutoTokenizer, AutoModel

    # Muat tokenizer & model (lazy — hanya sekali)
    global _bert_tokenizer, _bert_model, _bert_device
    if "_bert_tokenizer" not in globals():
        MODEL_NAME = "indobenchmark/indobert-base-p1"
        print(f"Memuat IndoBERT ({MODEL_NAME})...")
        _bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        _bert_model     = AutoModel.from_pretrained(MODEL_NAME).eval()
        _bert_device    = "cuda" if torch.cuda.is_available() else "cpu"
        _bert_model     = _bert_model.to(_bert_device)

    query_clean = remove_admin_identity_lines(query)
    inputs = _bert_tokenizer(
        query_clean, return_tensors="pt",
        truncation=True, max_length=512, padding=True
    ).to(_bert_device)

    with torch.no_grad():
        out = _bert_model(**inputs)

    mask   = inputs["attention_mask"].unsqueeze(-1)
    summed = (out.last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1)
    qvec   = (summed / counts).cpu().numpy()

    sims    = cosine_similarity(qvec, train_embeddings).flatten()
    top_idx = np.argsort(sims)[::-1][:k]
    return [(train_df.iloc[i]["case_id"], float(sims[i])) for i in top_idx]


print("✅ Fungsi retrieve_tfidf() dan retrieve_embedding() siap.")


✅ Fungsi retrieve_tfidf() dan retrieve_embedding() siap.


## 3. Ekstrak Solusi — Bangun `case_solutions`

Sesuai spesifikasi tugas: *"dari kasus top-k, ambil amar putusan atau ringkasan dakwaan
sebagai solusi"*. Struktur yang dibangun: `{case_id: solusi_text}`.

Solusi diambil dari kolom `argumen_hukum` (berisi inti amar putusan) yang dikombinasikan
dengan `pasal_terbukti` dan `vonis_penjara` agar ringkas dan informatif untuk CBR.


In [19]:
def build_case_solutions(df: pd.DataFrame) -> dict:
    """
    Bangun dictionary {case_id: solusi_text} dari DataFrame cases.
    Solusi = gabungan ringkas: pasal terbukti + vonis penjara + potongan argumen hukum.
    """
    solutions = {}
    for _, row in df.iterrows():
        cid = row["case_id"]

        # Komponen solusi
        pasal   = row.get("pasal_terbukti")  or row.get("pasal", "-")
        vonis   = row.get("vonis_penjara")   or "-"
        argumen = str(row.get("argumen_hukum") or "")[:400]   # maks 400 karakter

        solusi_text = (
            f"[PASAL TERBUKTI] {pasal}\n"
            f"[VONIS PENJARA]  {vonis}\n"
            f"[AMAR/ARGUMEN]   {argumen.strip()}"
        )
        solutions[cid] = solusi_text

    return solutions


case_solutions = build_case_solutions(cases_df)

print(f"✅ case_solutions dibangun untuk {len(case_solutions)} kasus.")
print()
# Tampilkan contoh
sample_id = train_df["case_id"].iloc[0]
print(f"Contoh solusi [{sample_id}]:")
print(case_solutions[sample_id])


✅ case_solutions dibangun untuk 30 kasus.

Contoh solusi [case_029]:
[PASAL TERBUKTI] Pasal 45; Pasal 28 ayat (1); Pasal 3; Pasal 5; Pasal 10; Pasal 378; Pasal 39; Pasal 40; Pasal 2 ayat (1); Pasal 28
[VONIS PENJARA]  -
[AMAR/ARGUMEN]   Perkara
Pidana pada Peradilan Tingkat Pertama, dengan acara pemeriksaan Biasa telah
menjatuhkan Putusan sebagai berikut dalam perkara Terdakwa :

Nama lengkap
: VERRY WU;
Tempat lahir
: Singkawang;
Umur / tanggal lahir : 24 tahun / 07 Mei 1988;
Jenis kelamin
: Laki-laki.
Kebangsaan
: Indonesia.
Tempat tinggal : Dusun Sukadamai Rt.02 Rw.01 Kel. Sungai Duri Kec.
Sungai Raya, Kab. Bengkayang Kalimant


## 4. Algoritma Prediksi

Dua strategi sesuai spesifikasi tugas:

| Strategi | Cara Kerja | Keunggulan |
|---|---|---|
| **Majority Vote** | Hitung pasal yang paling banyak muncul di top-k | Sederhana, robust terhadap outlier |
| **Weighted Similarity** | Bobot tiap kasus = skor cosine-similarity-nya | Memberi bobot lebih pada kasus yang paling mirip |


In [20]:
def predict_majority_vote(top_k_with_scores: list[tuple[str, float]]) -> dict:
    """
    Majority vote: pilih pasal yang paling banyak muncul di top-k.

    Args:
        top_k_with_scores: list (case_id, sim_score) dari retrieve()
    Returns:
        dict dengan predicted_pasal, vote_count, top_k_solutions
    """
    top_k_ids   = [cid for cid, _ in top_k_with_scores]
    pasal_list  = []

    for cid in top_k_ids:
        row = cases_df[cases_df["case_id"] == cid]
        if not row.empty:
            p = row.iloc[0].get("pasal_terbukti") or row.iloc[0].get("pasal", "UNKNOWN")
            pasal_list.append(str(p))

    if not pasal_list:
        return {"predicted_pasal": "UNKNOWN", "vote_count": 0, "top_k_solutions": {}}

    counter          = Counter(pasal_list)
    predicted_pasal  = counter.most_common(1)[0][0]
    vote_count       = counter.most_common(1)[0][1]

    return {
        "predicted_pasal"  : predicted_pasal,
        "vote_count"       : vote_count,
        "vote_distribution": dict(counter),
        "top_k_solutions"  : {cid: case_solutions.get(cid, "-") for cid in top_k_ids},
    }


def predict_weighted_similarity(top_k_with_scores: list[tuple[str, float]]) -> dict:
    """
    Weighted similarity: akumulasi skor similarity per pasal,
    pilih pasal dengan skor total tertinggi.

    Args:
        top_k_with_scores: list (case_id, sim_score) dari retrieve()
    Returns:
        dict dengan predicted_pasal, weighted_score, top_k_solutions
    """
    pasal_scores: dict[str, float] = {}
    top_k_ids    = [cid for cid, _ in top_k_with_scores]

    for cid, score in top_k_with_scores:
        row = cases_df[cases_df["case_id"] == cid]
        if not row.empty:
            p = row.iloc[0].get("pasal_terbukti") or row.iloc[0].get("pasal", "UNKNOWN")
            p = str(p)
            pasal_scores[p] = pasal_scores.get(p, 0.0) + score

    if not pasal_scores:
        return {"predicted_pasal": "UNKNOWN", "weighted_score": 0.0, "top_k_solutions": {}}

    predicted_pasal = max(pasal_scores, key=pasal_scores.__getitem__)
    return {
        "predicted_pasal"   : predicted_pasal,
        "weighted_score"    : round(pasal_scores[predicted_pasal], 4),
        "score_distribution": {k: round(v, 4) for k, v in
                               sorted(pasal_scores.items(), key=lambda x: -x[1])},
        "top_k_solutions"   : {cid: case_solutions.get(cid, "-") for cid in top_k_ids},
    }


print("✅ Algoritma majority_vote dan weighted_similarity siap.")


✅ Algoritma majority_vote dan weighted_similarity siap.


## 5. Implementasi Fungsi `predict_outcome()`

Sesuai spesifikasi tugas — wrapper tunggal yang menerima query teks dan mengembalikan
prediksi solusi (pasal + vonis + argumen), dengan pilihan jalur retrieval dan strategi voting.


In [21]:
def predict_outcome(
    query: str,
    k: int = 5,
    retrieval: str = "tfidf",    # "tfidf" | "embedding"
    strategy: str  = "weighted", # "majority" | "weighted"
) -> dict:
    """
    Prediksi solusi untuk kasus baru menggunakan CBR.

    Args:
        query     : teks kasus baru (ringkasan fakta / dakwaan)
        k         : jumlah kasus termirip yang dipertimbangkan
        retrieval : jalur retrieval ("tfidf" atau "embedding")
        strategy  : algoritma prediksi ("majority" atau "weighted")

    Returns:
        dict berisi:
          - predicted_pasal    : pasal yang diprediksi
          - predicted_solution : teks solusi lengkap dari kasus dengan pasal terprediksi
          - top_k_case_ids     : list case_id top-k
          - top_k_scores       : skor similarity tiap kasus
          - method_details     : detail vote/weight distribution
    """
    # 1) Retrieve top-k beserta skor similarity
    if retrieval == "embedding":
        top_k_with_scores = retrieve_embedding(query, k=k)
    else:
        top_k_with_scores = retrieve_tfidf(query, k=k)

    top_k_ids    = [cid for cid, _ in top_k_with_scores]
    top_k_scores = {cid: round(score, 4) for cid, score in top_k_with_scores}

    # 2) Prediksi pasal via voting/weighting
    if strategy == "majority":
        result = predict_majority_vote(top_k_with_scores)
    else:
        result = predict_weighted_similarity(top_k_with_scores)

    predicted_pasal = result["predicted_pasal"]

    # 3) Ambil teks solusi dari kasus dengan pasal yang diprediksi
    #    (kasus termirip di antara yang pasalnya = predicted_pasal)
    predicted_solution = "-"
    for cid in top_k_ids:
        row = cases_df[cases_df["case_id"] == cid]
        if not row.empty:
            p = row.iloc[0].get("pasal_terbukti") or row.iloc[0].get("pasal", "")
            if str(p) == predicted_pasal:
                predicted_solution = case_solutions.get(cid, "-")
                break

    return {
        "predicted_pasal"    : predicted_pasal,
        "predicted_solution" : predicted_solution,
        "top_k_case_ids"     : top_k_ids,
        "top_k_scores"       : top_k_scores,
        "method_details"     : result,
    }


print("✅ Fungsi predict_outcome() siap.")
print()
print("Signature:")
print("  predict_outcome(query, k=5, retrieval='tfidf'|'embedding', strategy='majority'|'weighted')")


✅ Fungsi predict_outcome() siap.

Signature:
  predict_outcome(query, k=5, retrieval='tfidf'|'embedding', strategy='majority'|'weighted')


## 6. Demo Manual — 5 Contoh Kasus Baru

Sesuai spesifikasi tugas: *"Siapkan 5 contoh kasus baru → jalankan `predict_outcome()` →
bandingkan dengan putusan sebenarnya."*

Kasus uji diambil dari **test set** (tidak ada di case base/train set) sehingga prediksi
benar-benar berdasarkan kasus lama yang mirip, bukan kasus itu sendiri.
Dibandingkan dua jalur retrieval (TF-IDF vs Embedding) dengan strategi weighted similarity.


In [24]:
# Muat queries.json untuk dapat ground truth
with open(EVAL_DIR / "queries.json", "r", encoding="utf-8") as f:
    queries_data = json.load(f)
    
# ── Patch kompatibilitas format lama → baru ─────────────────────────
for q in queries_data:
    if "_test_set_case_id" not in q:
        q["_test_set_case_id"] = q.get("ground_truth_case_id", "unknown")
    if "ground_truth_case_ids_same_pasal" not in q:
        q["ground_truth_case_ids_same_pasal"] = []

# Muat test_df dari cases yang bukan train (rekonstruksi test set)
train_ids  = set(train_df["case_id"])
test_df_full = cases_df[~cases_df["case_id"].isin(train_ids)].reset_index(drop=True)

demo_cases = queries_data[:5]  # ambil 5 query pertama

print("=" * 75)
print("DEMO PREDICT_OUTCOME — 5 KASUS BARU")
print("=" * 75)

demo_records = []

for q in demo_cases:
    query_text  = q["query_text"]
    gt_pasal    = q["ground_truth_pasal"]
    test_cid    = q["_test_set_case_id"]
    gt_ids      = q["ground_truth_case_ids_same_pasal"]

    # Prediksi dua jalur
    pred_tfidf = predict_outcome(query_text, k=5, retrieval="tfidf",     strategy="weighted")
    pred_bert  = predict_outcome(query_text, k=5, retrieval="embedding", strategy="weighted")

    # Cek apakah prediksi benar
    match_tfidf = (pred_tfidf["predicted_pasal"] == gt_pasal)
    match_bert  = (pred_bert["predicted_pasal"]  == gt_pasal)

    print(f"\n{'─'*70}")
    print(f"Query      : {q['query_id']}  (test case_id: {test_cid})")
    print(f"GT Pasal   : {gt_pasal}")
    print(f"GT same-pasal IDs di train: {gt_ids if gt_ids else '(tidak ada)'}")
    print()
    print(f"  [TF-IDF]   Prediksi pasal : {pred_tfidf['predicted_pasal']}  {'✅' if match_tfidf else '❌'}")
    print(f"             Top-5 IDs      : {pred_tfidf['top_k_case_ids']}")
    print(f"             Score dist.    : {pred_tfidf['method_details'].get('score_distribution', {})}")
    print()
    print(f"  [BERT]     Prediksi pasal : {pred_bert['predicted_pasal']}   {'✅' if match_bert else '❌'}")
    print(f"             Top-5 IDs      : {pred_bert['top_k_case_ids']}")
    print(f"             Score dist.    : {pred_bert['method_details'].get('score_distribution', {})}")

    demo_records.append({
        "query_id"              : q["query_id"],
        "test_case_id"          : test_cid,
        "ground_truth_pasal"    : gt_pasal,
        "pred_pasal_tfidf"      : pred_tfidf["predicted_pasal"],
        "pred_pasal_bert"       : pred_bert["predicted_pasal"],
        "match_tfidf"           : match_tfidf,
        "match_bert"            : match_bert,
    })

demo_df = pd.DataFrame(demo_records)
print(f"\n{'='*75}")
print("RINGKASAN DEMO:")
print(demo_df[["query_id","ground_truth_pasal","pred_pasal_tfidf","match_tfidf","pred_pasal_bert","match_bert"]].to_string(index=False))
acc_tfidf = demo_df["match_tfidf"].mean()
acc_bert  = demo_df["match_bert"].mean()
print(f"\nAkurasi demo TF-IDF  : {acc_tfidf:.0%} ({demo_df['match_tfidf'].sum()}/5 benar)")
print(f"Akurasi demo BERT    : {acc_bert:.0%}  ({demo_df['match_bert'].sum()}/5 benar)")


DEMO PREDICT_OUTCOME — 5 KASUS BARU


C:\Users\maull\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Memuat IndoBERT (indobenchmark/indobert-base-p1)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34497.48it/s]



──────────────────────────────────────────────────────────────────────
Query      : query_case_028  (test case_id: case_028)
GT Pasal   : Pasal 28
GT same-pasal IDs di train: (tidak ada)

  [TF-IDF]   Prediksi pasal : Pasal 45; Pasal 28 ayat (1); Pasal 3; Pasal 5; Pasal 10; Pasal 378; Pasal 39; Pasal 40; Pasal 2 ayat (1); Pasal 28  ❌
             Top-5 IDs      : ['case_029', 'case_030', 'case_019', 'case_001', 'case_007']
             Score dist.    : {'Pasal 45; Pasal 28 ayat (1); Pasal 3; Pasal 5; Pasal 10; Pasal 378; Pasal 39; Pasal 40; Pasal 2 ayat (1); Pasal 28': 0.5136, 'Pasal 303; Pasal 45; Pasal 27; Pasal 303 ayat (1); Pasal 6 ayat (2)': 0.4249, 'Pasal 27 ayat (4) Jo. Pasal 45 ayat (4)': 0.4031, 'Pasal 30 ayat (1) jo Pasal 46; Pasal 64 ayat (1); Pasal 3; Pasal 372; Pasal 2 ayat (1); Pasal 4; Pasal 5; Pasal 5 ayat (1); Pasal 5 ayat (2); Pasal 2': 0.3358, 'Pasal 28 ayat (1) Jo. Pasal 45; Pasal 55 ayat (1); Pasal 30 ayat (1) Jo. Pasal 46 ayat (1); Pasal 27 ayat (1); Pasal 1; Pas

## 7. Simpan Output — `data/results/predictions.csv`

Sesuai format yang diminta spesifikasi tugas:

| `query_id` | `predicted_solution` | `top_5_case_ids` |
|---|---|---|


In [25]:
all_records = []

for q in queries_data:
    query_text = q["query_text"]
    gt_pasal   = q["ground_truth_pasal"]
    test_cid   = q["_test_set_case_id"]

    # Jalankan predict_outcome dua jalur, dua strategi → simpan semua
    for retrieval in ["tfidf", "embedding"]:
        for strategy in ["majority", "weighted"]:
            pred = predict_outcome(query_text, k=5, retrieval=retrieval, strategy=strategy)

            all_records.append({
                "query_id"            : q["query_id"],
                "test_case_id"        : test_cid,
                "ground_truth_pasal"  : gt_pasal,
                "retrieval_method"    : retrieval,
                "prediction_strategy" : strategy,
                "predicted_pasal"     : pred["predicted_pasal"],
                "predicted_solution"  : pred["predicted_solution"][:300],  # ringkasan
                "top_5_case_ids"      : str(pred["top_k_case_ids"]),
                "top_5_scores"        : str(pred["top_k_scores"]),
                "correct"             : (pred["predicted_pasal"] == gt_pasal),
            })

predictions_df = pd.DataFrame(all_records)

# Simpan CSV
pred_path = RESULTS_DIR / "predictions.csv"
predictions_df.to_csv(pred_path, index=False)
print(f"✅ predictions.csv disimpan ke: {pred_path}")
print(f"   Total baris: {len(predictions_df)} ({len(queries_data)} query × 4 kombinasi metode)")
print()
print("Preview (weighted similarity, kedua jalur):")
preview = predictions_df[predictions_df["prediction_strategy"] == "weighted"][
    ["query_id", "retrieval_method", "predicted_pasal", "top_5_case_ids", "correct"]
]
print(preview.to_string(index=False))


✅ predictions.csv disimpan ke: C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\results\predictions.csv
   Total baris: 24 (6 query × 4 kombinasi metode)

Preview (weighted similarity, kedua jalur):
      query_id retrieval_method                                                                                                                                                                                                                                  predicted_pasal                                               top_5_case_ids  correct
query_case_028            tfidf                                                                                                                               Pasal 45; Pasal 28 ayat (1); Pasal 3; Pasal 5; Pasal 10; Pasal 378; Pasal 39; Pasal 40; Pasal 2 ayat (1); Pasal 28 ['case_029', 'case_030', 'case_019', 'case_001', 'case_007']    False
query_case_028        embedding                                            

## 8. Catatan

**Output Tahap 4 yang dihasilkan:**
- `case_solutions` — dictionary `{case_id: solusi_text}` untuk semua kasus
- Fungsi `predict_outcome()` dengan 2 jalur retrieval × 2 strategi voting
- Demo manual 5 kasus baru dengan perbandingan TF-IDF vs IndoBERT
- `data/results/predictions.csv` — semua prediksi tersimpan

**Interpretasi hasil demo:**
- Jika akurasi rendah pada dataset 30 dokumen → **wajar**, karena keterbatasan ukuran case base
- Perbedaan TF-IDF vs BERT menunjukkan mana representasi yang lebih cocok untuk domain UU ITE
- Kasus dengan `ground_truth_case_ids_same_pasal = []` sulit diprediksi benar karena  
  tidak ada kasus serupa di case base → highlight keterbatasan CBR pada kasus out-of-distribution

